# Analyse des erreurs — EmoBERT
**Mémoire M1 — Zinedine Hamadi & Sara Hadidi — Sorbonne Université 2025/2026**

Objectif : identifier les causes des mauvaises classifications pour proposer des pistes d'amélioration.

Pour chaque cas mal classé, on analyse :
- Émotion annotée vs prédite
- Ambiguïté du texte
- Chevauchement entre émotions
- Problèmes d'annotation
- Limite du contexte disponible

In [3]:
# ── Cellule 0 : Installation & imports ───────────────────────────────────────
!pip install -q transformers torch scikit-learn pandas odfpy

from google.colab import drive
drive.mount('/content/drive')

import os, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

TARGET_CLASSES = ['stress', 'frustration', 'engagement', 'confusion', 'satisfaction', 'neutre']
label2id = {c: i for i, c in enumerate(TARGET_CLASSES)}
id2label = {i: c for c, i in label2id.items()}
DRIVE    = '/content/drive/MyDrive/memoire_M1'
SEED     = 42

print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('✓ Imports OK')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU : Tesla T4
✓ Imports OK


In [4]:
# ── Cellule 1 : Charger le modèle et le corpus annoté ────────────────────────
import shutil

# Copier le modèle depuis Drive vers Colab localement
print('Copie du modèle depuis Drive...')
if os.path.exists('/content/emobert_v2'):
    shutil.rmtree('/content/emobert_v2')
shutil.copytree('/content/drive/MyDrive/emobert_v2', '/content/emobert_v2')
print('✓ Modèle copié localement')

MODEL_PATH = '/content/emobert_v2'   # ← chemin local
tokenizer  = AutoTokenizer.from_pretrained(MODEL_PATH)
model      = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()
print(f'✓ Modèle chargé sur {device}')

# Charger les annotations originales (sans GoEmotions)
df_z = pd.read_excel(f'{DRIVE}/zinedine_annotation.ods', engine='odf')
df_s = pd.read_excel(f'{DRIVE}/sara_annotation.ods',     engine='odf')
df   = pd.concat([df_z, df_s], ignore_index=True)
df   = df[df['edu_emotion'].notna()].copy()
df['edu_emotion'] = df['edu_emotion'].str.lower().str.strip()
df['edu_emotion'] = df['edu_emotion'].replace({
    'confision': 'confusion', 'neuret': 'neutre', 'neurtre': 'neutre'
})
df = df[df['edu_emotion'].isin(TARGET_CLASSES)]
df = df.drop_duplicates(subset=['utterance']).reset_index(drop=True)
print(f'✓ Corpus annoté : {len(df)} exemples')
print(df['edu_emotion'].value_counts().to_string())

Copie du modèle depuis Drive...
✓ Modèle copié localement


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ Modèle chargé sur cuda
✓ Corpus annoté : 494 exemples
edu_emotion
neutre          276
engagement      127
satisfaction     34
frustration      24
confusion        23
stress           10


In [5]:
# ── Cellule 2 : Prédire sur tout le corpus annoté ────────────────────────────
def predict(text: str) -> dict:
    inputs = tokenizer(
        text, return_tensors='pt',
        truncation=True, max_length=128, padding=True
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs      = F.softmax(logits, dim=-1).squeeze()
    confidence = probs.max().item()
    label_idx  = probs.argmax().item()
    label = TARGET_CLASSES[label_idx] if confidence >= 0.5 else 'neutre'
    return {
        'predicted': label,
        'confidence': round(confidence, 3),
        'all_scores': {c: round(p.item(), 3) for c, p in zip(TARGET_CLASSES, probs)}
    }

print('Prédiction sur tout le corpus...')
results = []
for _, row in df.iterrows():
    r = predict(row['utterance'])
    results.append({
        'utterance':    row['utterance'],
        'annotated':    row['edu_emotion'],
        'predicted':    r['predicted'],
        'confidence':   r['confidence'],
        'correct':      row['edu_emotion'] == r['predicted'],
        'all_scores':   str(r['all_scores'])
    })

df_results = pd.DataFrame(results)
accuracy   = df_results['correct'].mean()
print(f'\nAccuracy sur corpus annoté : {accuracy:.3f}')
print(f'Correct   : {df_results["correct"].sum()}')
print(f'Incorrect : {(~df_results["correct"]).sum()}')

Prédiction sur tout le corpus...

Accuracy sur corpus annoté : 0.530
Correct   : 262
Incorrect : 232


In [6]:
# ── Cellule 3 : Extraire les cas mal classés ──────────────────────────────────
errors = df_results[~df_results['correct']].copy().reset_index(drop=True)
print(f'Total erreurs : {len(errors)}\n')

print('=== Distribution des erreurs par classe annotée ===')
print(errors['annotated'].value_counts().to_string())
print()
print('=== Confusion annotated → predicted ===')
confusion = errors.groupby(['annotated', 'predicted']).size().reset_index(name='count')
confusion = confusion.sort_values('count', ascending=False)
print(confusion.to_string(index=False))

# Matrice de confusion
y_true = df_results['annotated'].map(label2id).tolist()
y_pred = df_results['predicted'].map(label2id).tolist()
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(TARGET_CLASSES))))
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=TARGET_CLASSES, yticklabels=TARGET_CLASSES, ax=ax)
ax.set_xlabel('Prédit'); ax.set_ylabel('Annoté')
ax.set_title('Matrice de confusion — corpus annoté')
plt.tight_layout()
plt.savefig(f'{DRIVE}/confusion_erreurs.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ confusion_erreurs.png sauvegardé')

Total erreurs : 232

=== Distribution des erreurs par classe annotée ===
annotated
neutre          185
engagement       19
frustration      12
confusion         7
satisfaction      7
stress            2

=== Confusion annotated → predicted ===
   annotated    predicted  count
      neutre   engagement     68
      neutre satisfaction     46
      neutre       stress     33
      neutre    confusion     19
      neutre  frustration     19
  engagement satisfaction      8
  engagement    confusion      4
   confusion   engagement      4
satisfaction   engagement      4
 frustration       stress      4
  engagement       stress      3
 frustration satisfaction      3
  engagement       neutre      2
   confusion       neutre      2
  engagement  frustration      2
 frustration   engagement      2
satisfaction    confusion      2
 frustration       neutre      2
   confusion       stress      1
 frustration    confusion      1
satisfaction       neutre      1
      stress   engagement     

In [7]:
# ── Cellule 4 : Tableau synthétique des erreurs ───────────────────────────────
# Prendre 5 erreurs représentatives par paire annotated→predicted
print('=== TABLEAU SYNTHÉTIQUE DES ERREURS ===\n')

# Prendre les paires d'erreurs les plus fréquentes
top_errors = confusion.head(10)

sample_errors = []
for _, row in top_errors.iterrows():
    ann, pred = row['annotated'], row['predicted']
    subset = errors[(errors['annotated'] == ann) & (errors['predicted'] == pred)]
    for _, e in subset.head(2).iterrows():
        sample_errors.append({
            'utterance': e['utterance'][:80] + '...' if len(e['utterance']) > 80 else e['utterance'],
            'annotated': ann,
            'predicted': pred,
            'confidence': f"{e['confidence']:.0%}",
        })

df_table = pd.DataFrame(sample_errors)
print(df_table.to_string(index=False))

# Sauvegarder
df_table.to_csv(f'{DRIVE}/erreurs_analyse.csv', index=False, encoding='utf-8')
errors.to_csv(f'{DRIVE}/tous_les_erreurs.csv',  index=False, encoding='utf-8')
print(f'\n✓ erreurs_analyse.csv sauvegardé ({len(df_table)} exemples)')
print(f'✓ tous_les_erreurs.csv sauvegardé ({len(errors)} exemples)')

=== TABLEAU SYNTHÉTIQUE DES ERREURS ===

                                                                          utterance    annotated    predicted confidence
                                                                 Do I need stamps ?       neutre   engagement        76%
                                                                Is that difficult ?       neutre   engagement        72%
It's ok. I bought tickets during matinee hours so it wasn't too bad. Yes, the po...       neutre satisfaction        59%
Nice! Mowing the lawn isn't fun in the heat unless you're using a super fast one...       neutre satisfaction        76%
   OK . Just think for a bit . I'll go help another customer . I'll be right back .       neutre       stress        62%
     Rock climbing is actually not scary unless you're afraid of heights of course.       neutre       stress        98%
                                                           oh i can understand that       neutre    confusion   

In [8]:
# ── Cellule 5 : Analyse qualitative automatique ───────────────────────────────
# Règles d'interprétation des erreurs les plus fréquentes
INTERPRETATIONS = {
    ('frustration', 'engagement'): (
        'Chevauchement sémantique',
        'Les phrases de frustration contiennent souvent des marqueurs '
        'de persévérance ("encore", "essayé") qui ressemblent à de l\'engagement actif.'
    ),
    ('frustration', 'neutre'): (
        'Contexte insuffisant',
        'Sans le contexte de la conversation précédente, '
        'le modèle ne peut pas détecter la frustration dans des phrases courtes.'
    ),
    ('stress', 'neutre'): (
        'Faible représentation',
        'La classe stress est très rare dans le corpus (10 exemples). '
        'Le modèle n\'a pas appris ses marqueurs spécifiques.'
    ),
    ('stress', 'frustration'): (
        'Chevauchement émotionnel',
        'Stress et frustration partagent des marqueurs linguistiques similaires '
        '("je n\'y arrive pas", "c\'est difficile"). La frontière est floue même pour les humains.'
    ),
    ('satisfaction', 'neutre'): (
        'Expressions implicites',
        'La satisfaction s\'exprime souvent de façon implicite ou sobre. '
        'Le modèle privilégie neutre quand les marqueurs positifs sont absents.'
    ),
    ('satisfaction', 'engagement'): (
        'Ambiguïté positive',
        'Les émotions positives (satisfaction, engagement) partagent '
        'un lexique similaire. La distinction nécessite le contexte complet.'
    ),
    ('confusion', 'neutre'): (
        'Questions neutres',
        'Certaines questions de compréhension semblent neutres sans contexte. '
        'Le modèle ne distingue pas question-curiosité et question-confusion.'
    ),
    ('engagement', 'neutre'): (
        'Engagement faible',
        'Un engagement peu marqué linguistiquement est confondu avec neutre. '
        'Les phrases courtes et directes manquent de marqueurs expressifs.'
    ),
}

DEFAULT_INTERPRETATION = (
    'Ambiguïté contextuelle',
    'Le texte ne contient pas assez de marqueurs explicites '
    'pour distinguer les deux émotions sans contexte conversationnel.'
)

print('=== ANALYSE QUALITATIVE DES ERREURS ===\n')
print(f'{"N°":<4} {"Utterance":<50} {"Annoté":<14} {"Prédit":<14} {"Cause":<25} Explication')
print('-' * 160)

analyse_rows = []
for i, row in df_table.iterrows():
    ann, pred = row['annotated'], row['predicted']
    cause, explication = INTERPRETATIONS.get(
        (ann, pred), DEFAULT_INTERPRETATION
    )
    print(f'{i+1:<4} {row["utterance"][:48]:<50} {ann:<14} {pred:<14} {cause:<25} {explication[:60]}')
    analyse_rows.append({
        'n': i+1,
        'utterance':    row['utterance'],
        'annotated':    ann,
        'predicted':    pred,
        'confidence':   row['confidence'],
        'cause':        cause,
        'explication':  explication
    })

df_analyse = pd.DataFrame(analyse_rows)
df_analyse.to_csv(f'{DRIVE}/analyse_qualitative.csv', index=False, encoding='utf-8')
print(f'\n✓ analyse_qualitative.csv sauvegardé')

=== ANALYSE QUALITATIVE DES ERREURS ===

N°   Utterance                                          Annoté         Prédit         Cause                     Explication
----------------------------------------------------------------------------------------------------------------------------------------------------------------
1    Do I need stamps ?                                 neutre         engagement     Ambiguïté contextuelle    Le texte ne contient pas assez de marqueurs explicites pour 
2    Is that difficult ?                                neutre         engagement     Ambiguïté contextuelle    Le texte ne contient pas assez de marqueurs explicites pour 
3    It's ok. I bought tickets during matinee hours s   neutre         satisfaction   Ambiguïté contextuelle    Le texte ne contient pas assez de marqueurs explicites pour 
4    Nice! Mowing the lawn isn't fun in the heat unle   neutre         satisfaction   Ambiguïté contextuelle    Le texte ne contient pas assez de marqueurs

In [9]:
# ── Cellule 6 : Synthèse des causes d'erreur ─────────────────────────────────
print('=== SYNTHÈSE DES CAUSES D\'ERREUR ===\n')

causes = df_analyse['cause'].value_counts()
print(causes.to_string())

# Graphique
fig, ax = plt.subplots(figsize=(9, 4))
causes.plot(kind='barh', ax=ax, color='#2E75B6')
ax.set_xlabel('Nombre d\'erreurs')
ax.set_title('Causes des erreurs de classification EmoBERT')
plt.tight_layout()
plt.savefig(f'{DRIVE}/causes_erreurs.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n=== PISTES D\'AMÉLIORATION ===')
print('''
1. Chevauchement sémantique (frustration/engagement, stress/frustration)
   → Redéfinir les frontières dans le guide d'annotation
   → Fusionner stress+frustration en une classe "émotions négatives"

2. Contexte insuffisant (utterances courtes sans historique)
   → Concaténer context+utterance au lieu d'utterance seule
   → Ajouter un window de 2-3 tours précédents

3. Faible représentation (stress : 10 exemples)
   → Collecter 50+ exemples de stress éducatif réel
   → Ou fusionner avec frustration

4. Expressions implicites (satisfaction sobre)
   → Enrichir le guide avec des exemples de satisfaction implicite
   → Réviser les annotations ambiguës avec les deux annotateurs
''')

print('✓ Analyse des erreurs terminée !')
print(f'Fichiers sauvegardés dans {DRIVE} :')
print('  - confusion_erreurs.png')
print('  - erreurs_analyse.csv')
print('  - tous_les_erreurs.csv')
print('  - analyse_qualitative.csv')
print('  - causes_erreurs.png')

=== SYNTHÈSE DES CAUSES D'ERREUR ===

cause
Ambiguïté contextuelle    18
Ambiguïté positive         2

=== PISTES D'AMÉLIORATION ===

1. Chevauchement sémantique (frustration/engagement, stress/frustration)
   → Redéfinir les frontières dans le guide d'annotation
   → Fusionner stress+frustration en une classe "émotions négatives"

2. Contexte insuffisant (utterances courtes sans historique)
   → Concaténer context+utterance au lieu d'utterance seule
   → Ajouter un window de 2-3 tours précédents

3. Faible représentation (stress : 10 exemples)
   → Collecter 50+ exemples de stress éducatif réel
   → Ou fusionner avec frustration

4. Expressions implicites (satisfaction sobre)
   → Enrichir le guide avec des exemples de satisfaction implicite
   → Réviser les annotations ambiguës avec les deux annotateurs

✓ Analyse des erreurs terminée !
Fichiers sauvegardés dans /content/drive/MyDrive/memoire_M1 :
  - confusion_erreurs.png
  - erreurs_analyse.csv
  - tous_les_erreurs.csv
  - analyse_